# Load Sample Source

Illustrate how to load a netCDF file for one source and plot it.

Also write it out as a GeoClaw dtopo file with ASCII raster format, as described in the [GeoClaw Documentation](https://www.clawpack.org/dtopo.html), and make some plots using GeoClaw tools.

Adapt this code to write it out to whatever format you need.

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from pathlib import Path

In [ ]:
dtopo_dir = './dtopofiles_nc'  # path to unzipped directory

In [ ]:
event = 'BL10D'
path = Path(dtopo_dir) / f'{event}.nc'
print(path)

## Open with xarray:

In [ ]:
import xarray as xr
with xr.open_dataset(path, decode_timedelta=False) as dtopo_xr:
    print(dtopo_xr)

## Open with rasterio:

In [ ]:
import rasterio
with rasterio.open(path) as src:
    print(f"Coordinate Reference System (CRS): {src.crs}")
    print(f"Bounds: {src.bounds}")
    print(f"Number of bands: {src.count}")
    print(f"Width/Height: {src.width}x{src.height}")
    
    meta = src.meta
    print('\nsrc.meta = \n')
    for k in meta.keys():
        print(f'{k:<30}{meta[k]}')
    
    tags = src.tags()
    print('\nsrc.tags = \n')
    for k in tags.keys():
        print(f'{k:<30}:  {tags[k]}')

## Load with GeoClaw

In [ ]:
from clawpack.geoclaw import dtopotools

dtopo = dtopotools.DTopography(path, dtopo_type=4)  # 4 ==> netcdf format

### Rewrite as GeoClaw ascii file

See [GeoClaw Documentation](https://www.clawpack.org/dtopo.html).

In [ ]:
fname = f'{event}.dtt3'
dtopo.write(fname, dtopo_type=3)

In [ ]:
sizeMB = Path(fname).stat().st_size / 1e6
print(f'{fname} has size {sizeMB:.1f} MB')

## Plots of deformation

Using the GeoClaw `dtopotools` module tools.

In [ ]:
coast = load('../topo/CSZ_coast.npy')  # precomputed coastline

In [ ]:
# time to plot deformation
#tplot = dtopo.times.max()  # for final static deformation
tplot1 = 200
tplot2 = 400

fig,axs = subplots(1,2,figsize=(10,6))
for ax in axs:
    ax.plot(coast[:,0], coast[:,1], 'g', linewidth=0.9)
    ax.set_aspect(1/cos(45*pi/180))
    ax.set_xlim(-130,-121)
    ax.set_ylim(39,50)

dtopo.plot_dZ_colors(t=tplot1, axes=axs[0], dZ_interval=100, cmax_dZ=10);
axs[0].set_title(f'Seafloor deformation dz\nat time t = {tplot1:.1f} seconds');

dtopo.plot_dZ_colors(t=tplot2, axes=axs[1], dZ_interval=100, cmax_dZ=10);
axs[1].set_title(f'Seafloor deformation dz\nat time t = {tplot2:.1f} seconds');